# Cell 1: Install dependencies

In [1]:
!pip install snntorch binary_fractions -q

# Cell 2: Mount drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Cell 3: Imports

In [3]:
import os, math, pickle, warnings
import numpy as np
import torch
from binary_fractions import Binary

warnings.filterwarnings('ignore')

# Cell 4: Paths & Hardware Config
**Change precision here — only these 3 values + OUTPUT_ROOT**

In [4]:
DATASET_DIR = '/content/drive/My Drive/To Your Path'
OUTPUT_ROOT = '/content/drive/My Drive/To Your Path/hw_q7_8'

# ── PRECISION CONFIG (change these 3 lines) ─────────────
integer_precision    = 7    # state integer bits
decimal_precision    = 8    # shared fractional bits
wt_integer_precision = 1    # weight integer bits
# ─────────────────────────────────────────────────────────

# Derived
PRECISION    = 1 + integer_precision + decimal_precision
WT_PRECISION = 1 + wt_integer_precision + decimal_precision

# Address encoding widths
layer_enc_bits       = 4
in_channel_enc_bits  = 8
out_channel_enc_bits = 8
neuron_enc_bits      = 16
fanin_enc_bits       = 12

# Network architecture
in_channels    = 3
c1             = 16
out_channels   = 32
kernel_size1   = 5
kernel_size2   = 5
stride1        = 2
stride2        = 5
seq_len        = 100
output_neurons = 3
num_steps      = 50
pad_samples    = 10

H_in  = (seq_len - kernel_size1) // stride1 + 1
H_out = (H_in - kernel_size2) // stride2 + 1
fc1_fanin  = out_channels * H_out
fc1_fanout = 64

# LIF parameters
vth = 1
decay_rate = 0.20
grow_rate = 1.0
vrest = 0
reset_mechanism = 1
refractory_period = 0

FOLDS = [1, 2, 3, 4, 5]

print(f'Hardware Config:')
print(f'  State:  Q{integer_precision}.{decimal_precision} = {PRECISION} bits')
print(f'  Weight: Q{wt_integer_precision}.{decimal_precision} = {WT_PRECISION} bits')
print(f'  Conv1: ({in_channels},{seq_len}) -> ({c1},{H_in})')
print(f'  Conv2: ({c1},{H_in}) -> ({out_channels},{H_out})')
print(f'  FC1: {fc1_fanin} -> {fc1_fanout}')
print(f'  FC2: {fc1_fanout} -> {output_neurons}')

# Cell 5: Quantization Functions & Address Encoding

In [ ]:
def fpmax(n, q):
    v = 0
    for i in range(n): v += 2 ** i
    for i in range(q): v += 1.0 / (2 ** (i + 1))
    return v

def TwosComplement(s):
    n = len(s)
    i = n - 1
    while i >= 0:
        if s[i] == '1': break
        i -= 1
    if i == -1: return '1' + s
    k = i - 1
    s = list(s)
    while k >= 0:
        s[k] = '0' if s[k] == '1' else '1'
        k -= 1
    return ''.join(s)

#### Truncating
# def fp2q(fp, weights=False):
#     n_fraction = decimal_precision
#     if weights:
#         n_integer = wt_integer_precision
#         fp_pos_max = fpmax(n=wt_integer_precision, q=n_fraction)
#     else:
#         n_integer = integer_precision
#         fp_pos_max = fpmax(n=integer_precision, q=n_fraction)
#     fp_pos_min = 0
#     fp_clip = np.clip(abs(fp), fp_pos_min, fp_pos_max)
#     fp = -fp_clip if fp < 0 else fp_clip
#     negative = False
#     if fp < 0:
#         fp = -fp
#         negative = True
#     (frac, dec) = math.modf(fp)
#     dec = int(dec)
#     ref_frac_str = ['0'] * n_fraction
#     if frac == 0: frac = 0.0001
#     frac_str = f"{Binary(frac)}"
#     for i in range(n_fraction):
#         try:
#             ref_frac_str[i] = frac_str[i + 4]
#         except IndexError:
#             ref_frac_str[i] = '0'
#     ref_frac_str = ''.join(ref_frac_str)
#     int_part = n_integer + 1
#     int_str = np.binary_repr(dec, width=int_part)
#     q_str = int_str + ref_frac_str
#     if negative:
#         q_str = TwosComplement(q_str)
#     return q_str

###### Rounding
def fp2q(fp, weights=False):
    n_fraction = decimal_precision
    if weights:
        n_integer = wt_integer_precision
        n_bits = wt_integer_precision + 1 + n_fraction
        fp_pos_max = fpmax(n=wt_integer_precision, q=n_fraction)
    else:
        n_integer = integer_precision
        n_bits = integer_precision + 1 + n_fraction
        fp_pos_max = fpmax(n=integer_precision, q=n_fraction)

    fp_clip = np.clip(fp, -fp_pos_max, fp_pos_max)
    q_int = int(round(fp_clip * (2 ** n_fraction)))

    max_val = (1 << (n_bits - 1)) - 1
    min_val = -(1 << (n_bits - 1))
    q_int = max(min_val, min(max_val, q_int))

    if q_int < 0:
        q_int = (1 << n_bits) + q_int

    q_str = format(q_int, f'0{n_bits}b')
    return q_str


def process_hex(value, nbits=32):
    if isinstance(value, str):
        n = int(value, 2)
    else:
        n = value
    n_hex = nbits // 4
    hex_val = hex(n)[2:]
    return hex_val.zfill(n_hex)

def address_encoding(layer=1, out_ch=16, in_ch=1, fanin=256):
    lhex    = process_hex(layer,  nbits=layer_enc_bits)
    o_chhex = process_hex(out_ch, nbits=in_channel_enc_bits)
    i_chhex = process_hex(in_ch,  nbits=out_channel_enc_bits)
    fhex    = process_hex(fanin,  nbits=fanin_enc_bits)
    return lhex + o_chhex + i_chhex + fhex

print(f'Quantization test:')
print(f'  vth=1.0     -> {fp2q(1.0)}')
print(f'  decay=0.20  -> {fp2q(0.20)} (actual={int(fp2q(0.20),2) / 2**decimal_precision:.4f})')
print(f'  wt=0.31     -> {fp2q(0.31, weights=True)}')
print(f'  wt=-0.31    -> {fp2q(-0.31, weights=True)}')

# Cell 6: Load inference data helper

In [ ]:
def load_inference_data(fold):
    path = os.path.join(DATASET_DIR, f'inference_hw_fold{fold}.pt')
    if not os.path.exists(path):
        print(f'  WARNING: {path} not found, skipping fold {fold}')
        return None
    data = torch.load(path, map_location='cpu')
    return data

def load_model_weights(fold):
    path = os.path.join(DATASET_DIR, f'fold{fold}_hw.pth')
    if not os.path.exists(path):
        print(f'  WARNING: {path} not found, skipping fold {fold}')
        return None
    return torch.load(path, map_location='cpu')

# Cell 7: Flush weights + biases
- Conv: `address_encoding(layer, out_ch, in_ch, fanin_idx)`
- FC: `address_encoding(layer, out_ch=0, in_ch=neuron, fanin=i)`
- Bias: `address_encoding(layer, out_ch=ch, in_ch=0, fanin=0)`
- All hex output: `nbits=32`

In [ ]:
def flush_conv_weights(weights_dict):
    addr = []
    data = []
    total_clipped = 0
    total_zeroed = 0
    wt_max_range = fpmax(wt_integer_precision, decimal_precision)
    conv_weights = [
        weights_dict['conv1.weight'],  # (16, 3, 5)
        weights_dict['conv2.weight'],  # (32, 16, 5)
    ]
    for layer in range(len(conv_weights)):
        wt_layer = conv_weights[layer]
        c_out, c_in, k_size = wt_layer.shape
        for out_ch in range(c_out):
            for in_ch in range(c_in):
                for fanin_idx in range(k_size):
                    raw = wt_layer[out_ch, in_ch, fanin_idx].item()
                    if abs(raw) > wt_max_range:
                        total_clipped += 1
                    processed_wt = fp2q(raw, weights=True)
                    if int(processed_wt, 2) == 0 and raw != 0:
                        total_zeroed += 1
                    data.append(process_hex(processed_wt, nbits=32))
                    addr.append(address_encoding(
                        layer=layer, out_ch=out_ch, in_ch=in_ch, fanin=fanin_idx
                    ))
    return addr, data, len(addr), total_clipped, total_zeroed


def flush_fc_weights(weights_dict, existing_addr, existing_data):
    addr = existing_addr
    data = existing_data
    total_clipped = 0
    total_zeroed = 0
    wt_max_range = fpmax(wt_integer_precision, decimal_precision)
    fc_layers = [
        (2, weights_dict['fc1.weight']),  # (64, 288)
        (3, weights_dict['fc2.weight']),  # (3, 64)
    ]
    for layer_id, wts in fc_layers:
        neurons = wts.shape[0]
        for neuron in range(neurons):
            wt_neuron = wts[neuron, :]
            for i, wt in enumerate(wt_neuron):
                raw = wt.item()
                if abs(raw) > wt_max_range:
                    total_clipped += 1
                processed_wt = fp2q(raw, weights=True)
                if int(processed_wt, 2) == 0 and raw != 0:
                    total_zeroed += 1
                data.append(process_hex(processed_wt, nbits=32))
                addr.append(address_encoding(
                    layer=layer_id, out_ch=0, in_ch=neuron, fanin=i
                ))
    return addr, data, total_clipped, total_zeroed


def flush_biases(weights_dict):
    bias_data = []
    bias_addr = []
    bias_layers = [
        (0, weights_dict['conv1.bias']),  # (16,)
        (1, weights_dict['conv2.bias']),  # (32,)
    ]
    for layer_id, bias in bias_layers:
        for ch in range(len(bias)):
            raw_b = bias[ch].item()
            q_bias = fp2q(raw_b, weights=False)
            bias_data.append(process_hex(q_bias, nbits=32))
            bias_addr.append(address_encoding(
                layer=layer_id, out_ch=ch, in_ch=0, fanin=0
            ))
    return bias_addr, bias_data, len(bias_data)


def flush_all_weights(weights_dict, out_dir):
    weight_dir = os.path.join(out_dir, 'weight')
    os.makedirs(weight_dir, exist_ok=True)
    # Weights
    addr, data, n_conv, conv_clip, conv_zero = flush_conv_weights(weights_dict)
    addr, data, fc_clip, fc_zero = flush_fc_weights(weights_dict, addr, data)
    with open(os.path.join(weight_dir, 'snncore.synaptic_weight.txt'), 'w') as f:
        for d in data: f.write(str(d) + '\n')
    with open(os.path.join(weight_dir, 'snncore.synaptic_address.txt'), 'w') as f:
        for a in addr: f.write(a + '\n')
    total_weights = len(addr)
    total_clipped = conv_clip + fc_clip
    total_zeroed = conv_zero + fc_zero
    # Biases
    b_addr, b_data, n_biases = flush_biases(weights_dict)
    with open(os.path.join(weight_dir, 'snncore.bias_values.txt'), 'w') as f:
        for d in b_data: f.write(str(d) + '\n')
    with open(os.path.join(weight_dir, 'snncore.bias_addresses.txt'), 'w') as f:
        for a in b_addr: f.write(a + '\n')
    print(f'    Conv weights: {n_conv}')
    print(f'    FC weights:   {total_weights - n_conv}')
    print(f'    Biases:       {n_biases}')
    return total_weights, total_clipped, total_zeroed

print('Flush weight functions defined.')

# Cell 8: Flush spike inputs + save functions

In [ ]:
def process_aer(aer, pad_samples=10):
    aer_data = aer
    if hasattr(aer, 'numpy'):
        aer_data = aer.detach().cpu().numpy()
    (time_steps, n_images, n_input) = aer_data.shape
    aer_pad = np.zeros((pad_samples, n_input))
    aer_out = np.zeros((pad_samples, n_input))
    for i in range(n_images):
        aer_out = np.concatenate((aer_out, aer_data[:, i, :], aer_pad), axis=0)
    return aer_out

def flush_aer(aer, pad_samples=10):
    aer_in = process_aer(aer, pad_samples)
    aer_hw = np.flip(aer_in, axis=1)
    return aer_hw

def flush_spike_inputs(inference_data, fold_dir):
    """Generate spike input files — one set per class (Normal, Moderate, Severe)."""
    input_dir = os.path.join(fold_dir, 'input')
    os.makedirs(input_dir, exist_ok=True)

    test_input = inference_data['input']
    actual_targets = inference_data['actual_targets'].numpy()
    N_test = test_input.shape[0]

    class_names_lower = ['normal', 'moderate', 'severe']
    CLASS_NAMES = ['Normal', 'Moderate', 'Severe']
    batch = 50
    total_batches = 0

    for cls in range(3):
        cls_mask = actual_targets == cls
        cls_indices = np.where(cls_mask)[0]
        N_cls = len(cls_indices)
        if N_cls == 0:
            continue

        cls_input = test_input[cls_indices]
        spike_data = cls_input.numpy()
        spike_flat = spike_data.reshape(N_cls, num_steps, -1)
        spike_aer = spike_flat.transpose(1, 0, 2)

        print(f'    {CLASS_NAMES[cls]}: {N_cls} samples')
        n_cls_batches = 0
        for strt_idx in range(0, N_cls, batch):
            end_idx = min(strt_idx + batch, N_cls)
            chunk = spike_aer[:, strt_idx:end_idx, :]
            aer_hw = flush_aer(chunk, pad_samples=pad_samples)
            out_path = os.path.join(input_dir,
                f'snncore.spikes_input_{class_names_lower[cls]}_{strt_idx}_{end_idx}.txt')
            np.savetxt(out_path, aer_hw, delimiter='', fmt='%d')
            n_cls_batches += 1
            total_batches += 1
            print(f'      Batch {n_cls_batches}: [{strt_idx}:{end_idx}] ({end_idx-strt_idx}), sim_cnt={aer_hw.shape[0]}')

    return total_batches, N_test

def save_torch_output(inference_data, fold_dir):
    """Save labels — per-class only."""
    output_dir = os.path.join(fold_dir, 'output')
    os.makedirs(output_dir, exist_ok=True)
    actual_targets = inference_data['actual_targets'].numpy()
    torch_targets = inference_data['torch_targets'].numpy()

    class_names_lower = ['normal', 'moderate', 'severe']
    for cls in range(3):
        cls_mask = actual_targets == cls
        cls_dict = {
            'actual_labels': actual_targets[cls_mask],
            'torch_labels': torch_targets[cls_mask],
            'num_steps': num_steps,
            'pad_samples': pad_samples,
            'n_test': cls_mask.sum(),
            'class': class_names_lower[cls],
            'original_indices': np.where(cls_mask)[0],
        }
        pickle.dump(cls_dict, open(os.path.join(output_dir,
            f'torch_out_{class_names_lower[cls]}.pkl'), 'wb'))

    return actual_targets, torch_targets

def save_parameters(fold_dir):
    param_dir = os.path.join(fold_dir, 'parameters')
    os.makedirs(param_dir, exist_ok=True)
    with open(os.path.join(param_dir, 'parameters.txt'), 'w') as f:
        f.write(f'// SpO2 — Q{integer_precision}.{decimal_precision} state / Q{wt_integer_precision}.{decimal_precision} weight\n')
        f.write(f'INTEGER_PRECISION = {integer_precision}\n')
        f.write(f'DECIMAL_PRECISION = {decimal_precision}\n')
        f.write(f'WT_INTEGER_PRECISION = {wt_integer_precision}\n')
        f.write(f'PRECISION = {PRECISION}\n')
        f.write(f'WT_PRECISION = {WT_PRECISION}\n')
        f.write(f'vth = {fp2q(vth)} // {vth}\n')
        f.write(f'decay_rate = {fp2q(decay_rate)} // {decay_rate}\n')
        f.write(f'grow_rate = {fp2q(grow_rate)} // {grow_rate}\n')
        f.write(f'vrest = {fp2q(vrest)} // {vrest}\n')
        f.write(f'reset_mechanism = {reset_mechanism} // {reset_mechanism}\n')
        f.write(f'refractory_period = {fp2q(refractory_period)} // {refractory_period}\n')

print('Spike flush and save functions defined.')

# Cell 9: Run all folds

In [ ]:
print(f'\n{"="*70}')
print(f'GENERATING HARDWARE FILES')
print(f'Q{integer_precision}.{decimal_precision} state ({PRECISION}b) / Q{wt_integer_precision}.{decimal_precision} weight ({WT_PRECISION}b)')
print(f'{"="*70}')

CLASS_NAMES = ['Normal', 'Moderate', 'Severe']
fold_summaries = []

for fold in FOLDS:
    print(f'\n{"─"*70}')
    print(f'FOLD {fold}')
    print(f'{"─"*70}')
    fold_dir = os.path.join(OUTPUT_ROOT, f'fold{fold}')
    os.makedirs(fold_dir, exist_ok=True)
    print(f'  Loading weights: fold{fold}_hw.pth')
    weights = load_model_weights(fold)
    if weights is None: continue
    print(f'  Loading inference data...')
    inference = load_inference_data(fold)
    if inference is None: continue
    N_test = inference['input'].shape[0]
    actual = inference['actual_targets'].numpy()
    torch_pred = inference['torch_targets'].numpy()
    print(f'  Samples: {N_test}')
    for cls in range(3):
        print(f'    {CLASS_NAMES[cls]}: {(actual == cls).sum()}')
    pt_acc = (torch_pred == actual).sum() / N_test * 100
    print(f'  PyTorch accuracy: {pt_acc:.1f}%')
    print(f'  Flushing weights (Q{wt_integer_precision}.{decimal_precision} = {WT_PRECISION}b)...')
    n_wt, n_clip, n_zero = flush_all_weights(weights, fold_dir)
    print(f'    Total weights: {n_wt}')
    print(f'    Clipped: {n_clip}')
    print(f'    Zeroed:  {n_zero}')
    print(f'  Flushing spike inputs...')
    n_batches, n_samples = flush_spike_inputs(inference, fold_dir)
    print(f'  Saving torch output...')
    save_torch_output(inference, fold_dir)
    save_parameters(fold_dir)
    fold_summaries.append({
        'fold': fold, 'n_test': N_test, 'pt_acc': pt_acc,
        'n_weights': n_wt, 'n_clipped': n_clip, 'n_zeroed': n_zero,
    })
    print(f'  Done -> {fold_dir}')

# Cell 10: Summary

In [ ]:
print(f'\n{"="*70}')
print(f'SUMMARY — Q{integer_precision}.{decimal_precision} state ({PRECISION}b) / '
      f'Q{wt_integer_precision}.{decimal_precision} weight ({WT_PRECISION}b)')
print(f'{"="*70}')
print(f'  {"Fold":<6} {"N":>5} {"PT Acc":>10} {"Clipped":>10} {"Zeroed":>10}')
print(f'  {"-"*42}')
for s in fold_summaries:
    print(f'  {s["fold"]:<6} {s["n_test"]:>5} {s["pt_acc"]:>9.1f}% {s["n_clipped"]:>10} {s["n_zeroed"]:>10}')
avg_pt = np.mean([s['pt_acc'] for s in fold_summaries])
print(f'\n  Average PyTorch accuracy: {avg_pt:.1f}%')
print(f'\n  RTL parameters:')
print(f'    .INTEGER_PRECISION({integer_precision}),')
print(f'    .DECIMAL_PRECISION({decimal_precision}),')
print(f'    .WT_INTEGER_PRECISION({wt_integer_precision}),')
print(f'\n  Output: {OUTPUT_ROOT}')

In [6]:
# ══════════════════════════════════════════════════════════════
# Cell 12: Read Hardware Output & Compare Accuracy
# ══════════════════════════════════════════════════════════════
import glob, re

# CONFIG — set which fold to read
READ_FOLD = 3  # change this
FOLD_DIR = os.path.join(OUTPUT_ROOT, f'fold{READ_FOLD}')

CLASS_NAMES = ['Normal', 'Moderate', 'Severe']
class_names_lower = ['normal', 'moderate', 'severe']
n_classes = 3

def process_spikes(filepath, reverse=True):
    """Read output spike file, count spikes per neuron per sample."""
    hwout = np.loadtxt(filepath, dtype=str)
    hwout_np = np.zeros((hwout.shape[0], output_neurons), dtype=int)
    for i, row in enumerate(hwout):
        bits = list(row)
        if reverse:
            bits = bits[::-1]
        for j in range(min(output_neurons, len(bits))):
            hwout_np[i, j] = int(bits[j])

    total_rows = hwout_np.shape[0]
    rows_per_sample = num_steps + pad_samples
    n_images = (total_rows - pad_samples) / rows_per_sample

    labels = []
    counts = []
    ties = []
    start_index = pad_samples

    for i in range(int(n_images)):
        end_index = start_index + num_steps + pad_samples
        sample_spikes = hwout_np[start_index:end_index]
        spike_counts = sample_spikes.sum(axis=0)

        max_count = spike_counts.max()
        winners = np.where(spike_counts == max_count)[0]
        is_tie = len(winners) > 1
        winner = winners[0]

        labels.append(winner)
        counts.append(spike_counts.tolist())
        ties.append(is_tie)
        start_index = end_index

    return np.array(labels), np.array(counts), np.array(ties)

# ══════════════════════════════════════════════════════════════
# Process each class
# ══════════════════════════════════════════════════════════════
output_dir = os.path.join(FOLD_DIR, 'output')

print(f'\n{"="*70}')
print(f'HARDWARE ACCURACY — FOLD {READ_FOLD}')
print(f'Q{integer_precision}.{decimal_precision} state ({PRECISION}b) / Q{wt_integer_precision}.{decimal_precision} weight ({WT_PRECISION}b)')
print(f'{"="*70}')

all_hw_correct = 0
all_hw_best_correct = 0
all_pt_correct = 0
all_total = 0
all_ties = 0
class_results = []

for cls in range(n_classes):
    cls_name = class_names_lower[cls]

    # Load torch labels
    pkl_path = os.path.join(output_dir, f'torch_out_{cls_name}.pkl')
    if not os.path.exists(pkl_path):
        print(f'\n  {CLASS_NAMES[cls]}: torch_out_{cls_name}.pkl not found, skipping')
        continue
    torch_data = pickle.load(open(pkl_path, 'rb'))
    actual_labels = torch_data['actual_labels']
    torch_labels = torch_data['torch_labels']
    N_cls = len(actual_labels)

    # Find hardware output spike files
    spk_pattern = os.path.join(output_dir, f'snncore.spikes_output_{cls_name}_*.txt')
    spk_files = sorted(glob.glob(spk_pattern))
    if not spk_files:
        spk_pattern = os.path.join(output_dir, f'*spikes_output*{cls_name}*.txt')
        spk_files = sorted(glob.glob(spk_pattern))
    if not spk_files:
        print(f'\n  {CLASS_NAMES[cls]}: No output spike files found, skipping')
        print(f'    Expected: snncore.spikes_output_{cls_name}_*.txt in {output_dir}')
        continue

    spk_files = sorted(spk_files,
        key=lambda f: int(re.search(r'_(\d+)_\d+', os.path.basename(f)).group(1))
        if re.search(r'_(\d+)_\d+', os.path.basename(f)) else 0)

    hw_labels_list, hw_counts_list, hw_ties_list = [], [], []
    for fpath in spk_files:
        labels, counts, ties = process_spikes(fpath, reverse=True)
        hw_labels_list.append(labels)
        hw_counts_list.append(counts)
        hw_ties_list.append(ties)

    hw_labels = np.concatenate(hw_labels_list)
    hw_counts = np.concatenate(hw_counts_list)
    hw_ties = np.concatenate(hw_ties_list)

    N = min(len(hw_labels), N_cls)
    actual = actual_labels[:N]
    pt_pred = torch_labels[:N]
    hw_pred = hw_labels[:N]
    counts_n = hw_counts[:N]
    ties_n = hw_ties[:N]

    pt_correct = (pt_pred == actual).sum()
    hw_correct = (hw_pred == actual).sum()
    n_ties = ties_n.sum()

    # Best-case ties
    hw_best = hw_pred.copy()
    for i in range(N):
        if ties_n[i]:
            max_count = counts_n[i].max()
            winners = np.where(counts_n[i] == max_count)[0]
            if actual[i] in winners:
                hw_best[i] = actual[i]
    hw_best_correct = (hw_best == actual).sum()

    print(f'\n  {CLASS_NAMES[cls]} ({N} samples, {len(spk_files)} files):')
    print(f'    Float32:          {pt_correct}/{N} = {100*pt_correct/N:.1f}%')
    print(f'    Hardware (argmax): {hw_correct}/{N} = {100*hw_correct/N:.1f}%')
    print(f'    Hardware (best):   {hw_best_correct}/{N} = {100*hw_best_correct/N:.1f}%')
    print(f'    Ties: {n_ties}/{N}')
    for n_idx in range(n_classes):
        c = counts_n[:, n_idx]
        print(f'    {CLASS_NAMES[n_idx]} neuron: mean={c.mean():.1f}, min={c.min()}, max={c.max()}')

    all_hw_correct += hw_correct
    all_hw_best_correct += hw_best_correct
    all_pt_correct += pt_correct
    all_total += N
    all_ties += n_ties
    class_results.append({
        'class': CLASS_NAMES[cls], 'n': N,
        'pt_acc': 100*pt_correct/N, 'hw_acc': 100*hw_correct/N,
        'hw_best_acc': 100*hw_best_correct/N, 'ties': n_ties,
    })

# ══════════════════════════════════════════════════════════════
# Overall summary
# ══════════════════════════════════════════════════════════════
if all_total > 0:
    print(f'\n{"="*70}')
    print(f'OVERALL — FOLD {READ_FOLD} ({all_total} samples)')
    print(f'{"="*70}')
    print(f'  Float32:          {all_pt_correct}/{all_total} = {100*all_pt_correct/all_total:.1f}%')
    print(f'  Hardware (argmax): {all_hw_correct}/{all_total} = {100*all_hw_correct/all_total:.1f}%')
    print(f'  Hardware (best):   {all_hw_best_correct}/{all_total} = {100*all_hw_best_correct/all_total:.1f}%')
    print(f'  Ties: {all_ties}/{all_total} ({100*all_ties/all_total:.1f}%)')
    print()
    print(f'  {"Class":<12} {"N":>5} {"Float32":>10} {"HW argmax":>12} {"HW best":>10} {"Ties":>6}')
    print(f'  {"-"*58}')
    for r in class_results:
        print(f'  {r["class"]:<12} {r["n"]:>5} {r["pt_acc"]:>9.1f}% {r["hw_acc"]:>11.1f}% '
              f'{r["hw_best_acc"]:>9.1f}% {r["ties"]:>6}')
    print(f'  {"-"*58}')
    print(f'  {"Total":<12} {all_total:>5} {100*all_pt_correct/all_total:>9.1f}% '
          f'{100*all_hw_correct/all_total:>11.1f}% '
          f'{100*all_hw_best_correct/all_total:>9.1f}% {all_ties:>6}')